# Module 5: Production Batch Evaluation

## Overview

In Module 4, we deployed observability-enabled agents that emit OTEL traces to CloudWatch. Now we need to **evaluate production traffic** at scale using the same evaluation criteria from Module 2.

This notebook implements a **batch evaluation pipeline** that:
1. **Sets up Infrastructure** - Creates Kinesis Firehose and S3 for trace streaming
2. **Exports** - Retrieves OTEL traces from agent runtime log groups
3. **Transforms** - Converts OTEL spans to evaluation test cases
4. **Evaluates** - Runs Strands eval (same evaluators as Module 2)
5. **Stores** - Saves results to S3 and CloudWatch metrics

## Architecture

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                   Production Batch Evaluation Pipeline                       │
│                                                                              │
│  ┌──────────────────┐                                                       │
│  │ Agent Runtime    │                                                       │
│  │  Log Groups      │                                                       │
│  │ (strands.tracer) │                                                       │
│  └────────┬─────────┘                                                       │
│           │                                                                  │
│           │  Subscription Filter                                            │
│           │  { $.scope.name = "strands.telemetry.tracer" &&                 │
│           │    $.body.output.messages[0].content.finish_reason = "end_turn" }│
│           ▼                                                                  │
│  ┌──────────────────┐     ┌──────────────────┐     ┌──────────────────┐    │
│  │ Kinesis Firehose │────▶│   S3 Bucket      │────▶│   Strands Eval   │    │
│  │ (traces-stream)  │     │ (traces/parquet) │     │   (batch job)    │    │
│  └──────────────────┘     └──────────────────┘     └────────┬─────────┘    │
│                                                              │              │
│                              ┌───────────────────────────────┼──────────┐   │
│                              │                               │          │   │
│                              ▼                               ▼          ▼   │
│                       ┌──────────┐              ┌──────────────┐  ┌─────────┐│
│                       │   S3     │              │  CloudWatch  │  │ Athena  ││
│                       │ Results  │              │   Metrics    │  │ Query   ││
│                       └──────────┘              └──────────────┘  └─────────┘│
└─────────────────────────────────────────────────────────────────────────────┘
```

## Key Insight: Agent Runtime Log Groups

**OTEL telemetry events are NOT in `aws/spans`**. They are in agent runtime log groups:
- `/aws/bedrock-agentcore/runtimes/{agent-id}-DEFAULT`
- The events have `scope.name = "strands.telemetry.tracer"`
- Final responses have `finish_reason: "end_turn"`

## Evaluation Metrics (Same as Module 2)

1. **Goal Success** - Did the agent successfully address the request?
2. **Helpfulness** - How helpful was the response?
3. **Routing Accuracy** - Was the request routed to the correct agent?
4. **Policy Compliance** - Did the response follow company policies?
5. **Response Quality** - Overall quality of the response
6. **Customer Satisfaction** - Predicted customer satisfaction

## Prerequisites

- ✅ Module 4a completed (observability agents deployed)
- ✅ Production traffic generated (invocations with traces)

### Time: ~25 minutes

## Step 1: Setup and Configuration

In [ ]:
# Install dependencies
!pip install -q boto3 strands-agents strands-agents-evals pandas

In [2]:
import boto3
import json
import os
import time
import pandas as pd
from datetime import datetime, timezone, timedelta
from typing import Dict, Any, List, Optional, Tuple
from botocore.exceptions import ClientError

# Try to load REGION from previous modules
try:
    %store -r REGION
    print(f"✅ Loaded REGION from previous module: {REGION}")
except:
    session = boto3.Session()
    REGION = session.region_name or 'us-west-2'
    print(f"Using session region: {REGION}")

# Initialize AWS clients
logs_client = boto3.client('logs', region_name=REGION)
s3_client = boto3.client('s3', region_name=REGION)
cloudwatch_client = boto3.client('cloudwatch', region_name=REGION)
sts_client = boto3.client('sts', region_name=REGION)
firehose_client = boto3.client('firehose', region_name=REGION)
iam_client = boto3.client('iam', region_name=REGION)

ACCOUNT_ID = sts_client.get_caller_identity()['Account']

# Configuration
TRACES_BUCKET = f'ecommerce-workshop-traces-{ACCOUNT_ID}-{REGION}'
RESULTS_BUCKET = f'ecommerce-workshop-eval-{ACCOUNT_ID}-{REGION}'
FIREHOSE_STREAM_NAME = 'ecommerce-workshop-traces-stream'
CLOUDWATCH_NAMESPACE = 'EcommerceWorkshop/BatchEvaluation'
LOOKBACK_HOURS = 24  # How far back to look for traces

print(f"\n📋 Configuration:")
print(f"   Account ID: {ACCOUNT_ID}")
print(f"   Region: {REGION}")
print(f"   Traces Bucket: {TRACES_BUCKET}")
print(f"   Results Bucket: {RESULTS_BUCKET}")
print(f"   Firehose Stream: {FIREHOSE_STREAM_NAME}")
print(f"   Lookback Hours: {LOOKBACK_HOURS}")

✅ Loaded REGION from previous module: us-west-2

📋 Configuration:
   Account ID: 537124949553
   Region: us-west-2
   Traces Bucket: ecommerce-workshop-traces-537124949553-us-west-2
   Results Bucket: ecommerce-workshop-eval-537124949553-us-west-2
   Firehose Stream: ecommerce-workshop-traces-stream
   Lookback Hours: 24


## Step 2: Set Up Streaming Infrastructure

Create the streaming pipeline components:
1. S3 bucket for traces
2. IAM role for Firehose
3. Kinesis Firehose delivery stream
4. CloudWatch Logs subscription filter

In [3]:
def create_s3_bucket(bucket_name: str) -> bool:
    """Create S3 bucket if it doesn't exist."""
    try:
        s3_client.head_bucket(Bucket=bucket_name)
        print(f"✅ S3 bucket exists: {bucket_name}")
        return True
    except ClientError:
        try:
            if REGION == 'us-east-1':
                s3_client.create_bucket(Bucket=bucket_name)
            else:
                s3_client.create_bucket(
                    Bucket=bucket_name,
                    CreateBucketConfiguration={'LocationConstraint': REGION}
                )
            print(f"✅ Created S3 bucket: {bucket_name}")
            return True
        except Exception as e:
            print(f"❌ Error creating bucket: {e}")
            return False


def create_firehose_role() -> str:
    """Create IAM role for Firehose to write to S3."""
    role_name = 'ecommerce-workshop-firehose-role'
    
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Principal": {"Service": "firehose.amazonaws.com"},
            "Action": "sts:AssumeRole"
        }]
    }
    
    s3_policy = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Action": [
                "s3:AbortMultipartUpload",
                "s3:GetBucketLocation",
                "s3:GetObject",
                "s3:ListBucket",
                "s3:ListBucketMultipartUploads",
                "s3:PutObject"
            ],
            "Resource": [
                f"arn:aws:s3:::{TRACES_BUCKET}",
                f"arn:aws:s3:::{TRACES_BUCKET}/*"
            ]
        }]
    }
    
    try:
        response = iam_client.get_role(RoleName=role_name)
        role_arn = response['Role']['Arn']
        print(f"✅ IAM role exists: {role_name}")
        return role_arn
    except iam_client.exceptions.NoSuchEntityException:
        try:
            response = iam_client.create_role(
                RoleName=role_name,
                AssumeRolePolicyDocument=json.dumps(trust_policy),
                Description='Role for Firehose to write traces to S3'
            )
            role_arn = response['Role']['Arn']
            
            iam_client.put_role_policy(
                RoleName=role_name,
                PolicyName='S3WritePolicy',
                PolicyDocument=json.dumps(s3_policy)
            )
            
            print(f"✅ Created IAM role: {role_name}")
            print("   Waiting 10s for role propagation...")
            time.sleep(10)
            return role_arn
        except Exception as e:
            print(f"❌ Error creating role: {e}")
            return ""


def create_cloudwatch_logs_role() -> str:
    """Create IAM role for CloudWatch Logs to write to Firehose."""
    role_name = 'ecommerce-workshop-cw-logs-role'
    
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Principal": {"Service": f"logs.{REGION}.amazonaws.com"},
            "Action": "sts:AssumeRole"
        }]
    }
    
    firehose_policy = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Action": [
                "firehose:PutRecord",
                "firehose:PutRecordBatch"
            ],
            "Resource": f"arn:aws:firehose:{REGION}:{ACCOUNT_ID}:deliverystream/{FIREHOSE_STREAM_NAME}"
        }]
    }
    
    try:
        response = iam_client.get_role(RoleName=role_name)
        role_arn = response['Role']['Arn']
        print(f"✅ CloudWatch Logs role exists: {role_name}")
        return role_arn
    except iam_client.exceptions.NoSuchEntityException:
        try:
            response = iam_client.create_role(
                RoleName=role_name,
                AssumeRolePolicyDocument=json.dumps(trust_policy),
                Description='Role for CloudWatch Logs to write to Firehose'
            )
            role_arn = response['Role']['Arn']
            
            iam_client.put_role_policy(
                RoleName=role_name,
                PolicyName='FirehoseWritePolicy',
                PolicyDocument=json.dumps(firehose_policy)
            )
            
            print(f"✅ Created CloudWatch Logs role: {role_name}")
            print("   Waiting 10s for role propagation...")
            time.sleep(10)
            return role_arn
        except Exception as e:
            print(f"❌ Error creating role: {e}")
            return ""


def create_firehose_stream(role_arn: str) -> bool:
    """Create Kinesis Firehose delivery stream."""
    try:
        firehose_client.describe_delivery_stream(
            DeliveryStreamName=FIREHOSE_STREAM_NAME
        )
        print(f"✅ Firehose stream exists: {FIREHOSE_STREAM_NAME}")
        return True
    except firehose_client.exceptions.ResourceNotFoundException:
        try:
            firehose_client.create_delivery_stream(
                DeliveryStreamName=FIREHOSE_STREAM_NAME,
                DeliveryStreamType='DirectPut',
                ExtendedS3DestinationConfiguration={
                    'RoleARN': role_arn,
                    'BucketARN': f'arn:aws:s3:::{TRACES_BUCKET}',
                    'Prefix': 'traces/year=!{timestamp:yyyy}/month=!{timestamp:MM}/day=!{timestamp:dd}/hour=!{timestamp:HH}/',
                    'ErrorOutputPrefix': 'errors/',
                    'BufferingHints': {
                        'SizeInMBs': 5,
                        'IntervalInSeconds': 60
                    },
                    'CompressionFormat': 'GZIP',
                    'CloudWatchLoggingOptions': {
                        'Enabled': False
                    }
                }
            )
            print(f"✅ Created Firehose stream: {FIREHOSE_STREAM_NAME}")
            print("   Waiting 30s for stream to become active...")
            time.sleep(30)
            return True
        except Exception as e:
            print(f"❌ Error creating Firehose: {e}")
            return False


print("="*60)
print("SETTING UP STREAMING INFRASTRUCTURE")
print("="*60)

# Create S3 buckets
create_s3_bucket(TRACES_BUCKET)
create_s3_bucket(RESULTS_BUCKET)

# Create IAM roles
FIREHOSE_ROLE_ARN = create_firehose_role()
CW_LOGS_ROLE_ARN = create_cloudwatch_logs_role()

# Create Firehose stream
if FIREHOSE_ROLE_ARN:
    create_firehose_stream(FIREHOSE_ROLE_ARN)

print("\n✅ Infrastructure setup complete")

SETTING UP STREAMING INFRASTRUCTURE
✅ S3 bucket exists: ecommerce-workshop-traces-537124949553-us-west-2
✅ S3 bucket exists: ecommerce-workshop-eval-537124949553-us-west-2
✅ IAM role exists: ecommerce-workshop-firehose-role
✅ CloudWatch Logs role exists: ecommerce-workshop-cw-logs-role
✅ Firehose stream exists: ecommerce-workshop-traces-stream

✅ Infrastructure setup complete


## Step 3: Create Subscription Filter for Agent Log Groups

The OTEL telemetry events are in agent runtime log groups. We need to:
1. Find the agent runtime log groups
2. Create subscription filters to stream traces to Firehose

In [4]:
def find_agent_runtime_log_groups() -> List[str]:
    """Find all AgentCore runtime log groups."""
    log_groups = []
    paginator = logs_client.get_paginator('describe_log_groups')
    
    for page in paginator.paginate(logGroupNamePrefix='/aws/bedrock-agentcore/runtimes/'):
        for lg in page.get('logGroups', []):
            log_groups.append(lg['logGroupName'])
    
    return log_groups


def create_subscription_filter(log_group_name: str, cw_logs_role_arn: str) -> bool:
    """
    Create CloudWatch Logs subscription filter for OTEL events.
    
    Filter pattern matches:
    - scope.name = "strands.telemetry.tracer" (Strands OTEL events)
    - finish_reason = "end_turn" (Final responses only)
    """
    filter_name = 'eval-traces-filter'
    
    # Filter for strands.telemetry.tracer events with end_turn finish_reason
    filter_pattern = '{ $.scope.name = "strands.telemetry.tracer" && $.body.output.messages[0].content.finish_reason = "end_turn" }'
    
    try:
        # Check if filter already exists
        response = logs_client.describe_subscription_filters(
            logGroupName=log_group_name,
            filterNamePrefix=filter_name
        )
        
        if response.get('subscriptionFilters'):
            print(f"   ✅ Subscription filter exists for {log_group_name}")
            return True
    except Exception:
        pass
    
    try:
        firehose_arn = f"arn:aws:firehose:{REGION}:{ACCOUNT_ID}:deliverystream/{FIREHOSE_STREAM_NAME}"
        
        logs_client.put_subscription_filter(
            logGroupName=log_group_name,
            filterName=filter_name,
            filterPattern=filter_pattern,
            destinationArn=firehose_arn,
            roleArn=cw_logs_role_arn
        )
        print(f"   ✅ Created subscription filter for {log_group_name}")
        return True
    except Exception as e:
        print(f"   ⚠️ Error creating filter for {log_group_name}: {str(e)[:80]}")
        return False


print("="*60)
print("SETTING UP LOG SUBSCRIPTION FILTERS")
print("="*60)

# Find agent runtime log groups
AGENT_LOG_GROUPS = find_agent_runtime_log_groups()
print(f"\nFound {len(AGENT_LOG_GROUPS)} agent runtime log groups:")
for lg in AGENT_LOG_GROUPS:
    print(f"   • {lg}")

# Create subscription filters
if AGENT_LOG_GROUPS and CW_LOGS_ROLE_ARN:
    print(f"\nCreating subscription filters...")
    for log_group in AGENT_LOG_GROUPS:
        create_subscription_filter(log_group, CW_LOGS_ROLE_ARN)
else:
    print("\n⚠️ No agent log groups found or IAM role not created")
    print("   Ensure agents from Module 4a are deployed")

SETTING UP LOG SUBSCRIPTION FILTERS

Found 34 agent runtime log groups:
   • /aws/bedrock-agentcore/runtimes/agent-7ke2kD9vfP-DEFAULT
   • /aws/bedrock-agentcore/runtimes/browser_agent-rc9J1PCdrR-DEFAULT
   • /aws/bedrock-agentcore/runtimes/claude_agent_sdk_demo-GjII0Z2pl7-DEFAULT
   • /aws/bedrock-agentcore/runtimes/claude_agent_sdk_demo-Nru6SN4s2P-DEFAULT
   • /aws/bedrock-agentcore/runtimes/claude_agent_sdk_demo-nCT2Gl5oGC-DEFAULT
   • /aws/bedrock-agentcore/runtimes/claude_agent_sdk_demo-z439Wl9rw4-DEFAULT
   • /aws/bedrock-agentcore/runtimes/ecommerce-workshop
   • /aws/bedrock-agentcore/runtimes/ecommerce_customer_service-j10dRbGkJf-DEFAULT
   • /aws/bedrock-agentcore/runtimes/ecommerce_workshop_account_agent-XlHlWO4MaP-DEFAULT
   • /aws/bedrock-agentcore/runtimes/ecommerce_workshop_account_agent_notrace-tfVTl1208v-DEFAULT
   • /aws/bedrock-agentcore/runtimes/ecommerce_workshop_account_agent_observability-BxEL2C599H-DEFAULT
   • /aws/bedrock-agentcore/runtimes/ecommerce_workshop_

## Step 4: OTEL Trace Transformer

This module transforms OTEL events into evaluation test cases.

**Key Insight: Tool Calls Are in a Different Scope**

OTEL events for a single trace span multiple scopes:
- **`strands.telemetry.tracer`**: Final response with `finish_reason: end_turn`
- **`opentelemetry.instrumentation.botocore.bedrock-runtime`**: Actual tool calls with `event.name: gen_ai.assistant.message`

Both share the same `traceId`, so we query ALL events for each trace to get:
1. The final response (input/output)
2. The actual tool calls (e.g., `OrderTools___get_order_status`)

**Tool Naming Convention:**
- `OrderTools___get_order_status` → ORDER domain
- `ProductTools___search_products` → PRODUCT domain
- `AccountTools___get_customer_info` → ACCOUNT domain

In [6]:
def parse_otel_event(message: str) -> Optional[Dict[str, Any]]:
    """
    Parse an OTEL event from a CloudWatch log message.
    """
    try:
        event = json.loads(message)
        
        scope_name = event.get('scope', {}).get('name', '')
        if scope_name != 'strands.telemetry.tracer':
            return None
        
        body = event.get('body', {})
        if not body or not isinstance(body, dict):
            return None
        
        if 'input' not in body or 'output' not in body:
            return None
        
        return event
        
    except (json.JSONDecodeError, TypeError):
        return None


def extract_user_message(input_messages: List[Dict]) -> str:
    """
    Extract the user message from OTEL input messages.
    """
    for msg in input_messages:
        role = msg.get('role', '')
        if role == 'user':
            content = msg.get('content', {})
            
            if isinstance(content, str):
                return content
            elif isinstance(content, dict):
                inner_content = content.get('content', '')
                if inner_content:
                    try:
                        parsed = json.loads(inner_content)
                        if isinstance(parsed, list) and len(parsed) > 0:
                            for item in parsed:
                                if isinstance(item, dict) and 'text' in item:
                                    return item['text']
                            return inner_content
                    except (json.JSONDecodeError, TypeError):
                        return inner_content
                
                text = content.get('text', '')
                if text:
                    return text
    
    return ''


def extract_assistant_response(output_messages: List[Dict]) -> str:
    """
    Extract assistant response text from OTEL output messages.
    """
    response_text = ''
    
    for msg in output_messages:
        role = msg.get('role', '')
        if role != 'assistant':
            continue
        
        content = msg.get('content', {})
        if not isinstance(content, dict):
            continue
        
        message_str = content.get('message', '')
        if not message_str:
            continue
        
        try:
            message_array = json.loads(message_str)
            if not isinstance(message_array, list):
                response_text = message_str
                continue
            
            text_parts = []
            for item in message_array:
                if isinstance(item, dict):
                    if 'text' in item:
                        text_parts.append(item['text'])
                elif isinstance(item, str):
                    text_parts.append(item)
            
            if text_parts:
                response_text = '\n'.join(text_parts)
                
        except (json.JSONDecodeError, TypeError):
            response_text = message_str
    
    return response_text


def is_final_response(event: Dict[str, Any]) -> bool:
    """
    Check if this OTEL event represents the final response.
    """
    body = event.get('body', {})
    output = body.get('output', {})
    messages = output.get('messages', [])
    
    for msg in messages:
        content = msg.get('content', {})
        if isinstance(content, dict):
            finish_reason = content.get('finish_reason', '')
            if finish_reason == 'end_turn':
                return True
    
    return False


def extract_tool_calls_from_trace_events(events: List[Dict]) -> List[Dict]:
    """
    Extract actual tool calls from all events in a trace.
    
    Tool calls are found in events with:
    - scope: opentelemetry.instrumentation.botocore.bedrock-runtime
    - event.name: gen_ai.assistant.message
    - body.tool_calls: array of tool call objects
    """
    tool_calls = []
    
    for event in events:
        try:
            data = json.loads(event) if isinstance(event, str) else event
            
            scope = data.get('scope', {}).get('name', '')
            event_name = data.get('attributes', {}).get('event.name', '')
            
            # Tool calls are in bedrock-runtime scope with gen_ai.assistant.message event
            if 'botocore.bedrock-runtime' in scope and event_name == 'gen_ai.assistant.message':
                body = data.get('body', {})
                
                # Extract from tool_calls array
                for tc in body.get('tool_calls', []):
                    func = tc.get('function', {})
                    tool_name = func.get('name', '')
                    tool_args = func.get('arguments', {})
                    
                    if tool_name:
                        tool_calls.append({
                            'name': tool_name,
                            'input': tool_args,
                            'tool_id': tc.get('id', ''),
                            'type': 'tool_call',
                        })
                
                # Also check content array for toolUse
                for content_item in body.get('content', []):
                    if isinstance(content_item, dict) and 'toolUse' in content_item:
                        tool_use = content_item['toolUse']
                        tool_name = tool_use.get('name', '')
                        if tool_name and not any(t['name'] == tool_name for t in tool_calls):
                            tool_calls.append({
                                'name': tool_name,
                                'input': tool_use.get('input', {}),
                                'tool_id': tool_use.get('toolUseId', ''),
                                'type': 'tool_call',
                            })
                            
        except (json.JSONDecodeError, TypeError, AttributeError):
            continue
    
    return tool_calls


def identify_routing_from_tools(tools: List[Dict]) -> Dict[str, Any]:
    """
    Identify the routing domain from actual tool calls.
    
    Tool naming convention: {ToolClass}___{method_name}
    Examples:
    - OrderTools___get_order_status → ORDER
    - ProductTools___search_products → PRODUCT  
    - AccountTools___get_customer_info → ACCOUNT
    """
    routing_info = {
        'primary_domain': 'UNKNOWN',
        'tools_used': [],
        'sub_agents_called': [],
    }
    
    domains_found = set()
    
    for tool in tools:
        tool_name = tool.get('name', '')
        routing_info['tools_used'].append(tool_name)
        
        # Parse tool name to identify domain
        tool_lower = tool_name.lower()
        
        # Check tool class prefix (e.g., OrderTools___, ProductTools___)
        if 'order' in tool_lower:
            domains_found.add('ORDER')
        elif 'product' in tool_lower:
            domains_found.add('PRODUCT')
        elif 'account' in tool_lower or 'customer' in tool_lower:
            domains_found.add('ACCOUNT')
        
        # Check for sub-agent patterns
        if 'agent' in tool_lower:
            routing_info['sub_agents_called'].append(tool_name)
    
    # Remove duplicates
    routing_info['tools_used'] = list(dict.fromkeys(routing_info['tools_used']))
    
    # Determine primary domain
    if len(domains_found) == 1:
        routing_info['primary_domain'] = list(domains_found)[0]
    elif len(domains_found) > 1:
        routing_info['primary_domain'] = 'MULTI'
    
    return routing_info


def get_all_trace_events(log_group: str, trace_id: str, start_time: int, end_time: int) -> List[Dict]:
    """
    Query all events for a specific trace ID to get tool calls.
    """
    try:
        response = logs_client.filter_log_events(
            logGroupName=log_group,
            filterPattern=f'{{ $.traceId = "{trace_id}" }}',
            startTime=start_time,
            endTime=end_time,
            limit=100
        )
        
        events = []
        for event in response.get('events', []):
            try:
                data = json.loads(event.get('message', ''))
                events.append(data)
            except:
                pass
        
        return events
    except Exception as e:
        return []


def transform_otel_to_test_case(
    event: Dict[str, Any], 
    tool_calls: List[Dict] = None
) -> Optional[Dict[str, Any]]:
    """
    Transform an OTEL event into an evaluation test case.
    
    Args:
        event: The strands.telemetry.tracer event with final response
        tool_calls: List of tool calls extracted from the full trace
    """
    try:
        body = event.get('body', {})
        attributes = event.get('attributes', {})
        
        input_data = body.get('input', {})
        input_messages = input_data.get('messages', [])
        
        output_data = body.get('output', {})
        output_messages = output_data.get('messages', [])
        
        user_message = extract_user_message(input_messages)
        if not user_message:
            return None
        
        assistant_response = extract_assistant_response(output_messages)
        if not assistant_response:
            return None
        
        # Use provided tool calls or empty list
        all_tools = tool_calls if tool_calls else []
        
        # Identify routing from actual tool calls
        routing_info = identify_routing_from_tools(all_tools)
        
        trace_id = event.get('traceId', '')
        span_id = event.get('spanId', '')
        session_id = attributes.get('session.id', '')
        
        time_unix_nano = event.get('timeUnixNano', 0)
        if time_unix_nano:
            timestamp = datetime.fromtimestamp(
                time_unix_nano / 1e9, tz=timezone.utc
            ).isoformat().replace('+00:00', 'Z')
        else:
            timestamp = datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z')
        
        return {
            'trace_id': trace_id,
            'span_id': span_id,
            'session_id': session_id,
            'input': user_message,
            'actual_output': assistant_response,
            'tools_called': all_tools,
            'routing_info': routing_info,
            'timestamp': timestamp,
        }
        
    except Exception as e:
        print(f"Error transforming event: {e}")
        return None


print("✅ OTEL Transformer functions defined")
print("   • parse_otel_event - Parse strands.telemetry.tracer events")
print("   • extract_user_message - Extract user input from messages")
print("   • extract_assistant_response - Extract final assistant response")
print("   • is_final_response - Check for finish_reason: end_turn")
print("   • extract_tool_calls_from_trace_events - Extract tools from bedrock-runtime events")
print("   • identify_routing_from_tools - Determine domain from tool names")
print("   • get_all_trace_events - Query all events for a trace ID")
print("   • transform_otel_to_test_case - Convert to evaluation format")
print("")
print("📋 Key insight: Tool calls are in a DIFFERENT scope than final responses!")
print("   • Final response: scope='strands.telemetry.tracer'")
print("   • Tool calls: scope='opentelemetry.instrumentation.botocore.bedrock-runtime'")
print("   • Both share the same traceId, so we query all events per trace")

✅ OTEL Transformer functions defined
   • parse_otel_event - Parse strands.telemetry.tracer events
   • extract_user_message - Extract user input from messages
   • extract_assistant_response - Extract final assistant response
   • is_final_response - Check for finish_reason: end_turn
   • extract_tool_calls_from_trace_events - Extract tools from bedrock-runtime events
   • identify_routing_from_tools - Determine domain from tool names
   • get_all_trace_events - Query all events for a trace ID
   • transform_otel_to_test_case - Convert to evaluation format

📋 Key insight: Tool calls are in a DIFFERENT scope than final responses!
   • Final response: scope='strands.telemetry.tracer'
   • Tool calls: scope='opentelemetry.instrumentation.botocore.bedrock-runtime'
   • Both share the same traceId, so we query all events per trace


## Step 5: Export Traces from Agent Runtime Log Groups

Query the agent runtime log groups directly to retrieve production traces.

In [7]:
def export_traces_from_log_groups(
    log_groups: List[str],
    lookback_hours: int = 24,
    max_traces: int = 100,
    fetch_tool_calls: bool = True
) -> List[Dict[str, Any]]:
    """
    Export OTEL traces from agent runtime log groups.
    
    This function:
    1. Finds final response events (strands.telemetry.tracer with finish_reason: end_turn)
    2. For each trace, queries ALL events to extract tool calls
    3. Tool calls are in 'opentelemetry.instrumentation.botocore.bedrock-runtime' scope
    
    Args:
        log_groups: List of log group names to query
        lookback_hours: How far back to look for traces
        max_traces: Maximum number of traces to return
        fetch_tool_calls: If True, query all events per trace to get tool calls
        
    Returns:
        List of test case dicts ready for evaluation
    """
    print(f"\nExporting traces from agent runtime log groups...")
    print(f"  Log groups: {len(log_groups)}")
    print(f"  Lookback: {lookback_hours} hours")
    print(f"  Max traces: {max_traces}")
    print(f"  Fetch tool calls: {fetch_tool_calls}")
    
    end_time = int(datetime.now(timezone.utc).timestamp() * 1000)
    start_time = int((datetime.now(timezone.utc) - timedelta(hours=lookback_hours)).timestamp() * 1000)
    
    test_cases = []
    trace_ids_seen = set()
    
    for log_group in log_groups:
        if len(test_cases) >= max_traces:
            break
        
        try:
            # First, filter for final response events
            response = logs_client.filter_log_events(
                logGroupName=log_group,
                filterPattern='{ $.scope.name = "strands.telemetry.tracer" && $.body.output.messages[0].content.finish_reason = "end_turn" }',
                startTime=start_time,
                endTime=end_time,
                limit=50
            )
            
            events = response.get('events', [])
            if not events:
                continue
            
            print(f"\n  {log_group.split('/')[-1]}: {len(events)} final response events")
            
            for event in events:
                if len(test_cases) >= max_traces:
                    break
                
                message = event.get('message', '')
                
                # Parse the OTEL event
                otel_event = parse_otel_event(message)
                if not otel_event:
                    continue
                
                # Avoid duplicates
                trace_id = otel_event.get('traceId', '')
                if trace_id in trace_ids_seen:
                    continue
                trace_ids_seen.add(trace_id)
                
                # Fetch all events for this trace to get tool calls
                tool_calls = []
                if fetch_tool_calls and trace_id:
                    all_trace_events = get_all_trace_events(
                        log_group, trace_id, start_time, end_time
                    )
                    tool_calls = extract_tool_calls_from_trace_events(all_trace_events)
                
                # Transform to test case with tool calls
                test_case = transform_otel_to_test_case(otel_event, tool_calls)
                if test_case:
                    test_cases.append(test_case)
                    
                    # Show progress for traces with tools
                    if tool_calls:
                        tools_str = ', '.join([t['name'] for t in tool_calls[:3]])
                        if len(tool_calls) > 3:
                            tools_str += f" (+{len(tool_calls)-3} more)"
                        print(f"    ✓ Trace {trace_id[:12]}: {len(tool_calls)} tools [{tools_str}]")
                
        except Exception as e:
            print(f"  Error querying {log_group.split('/')[-1]}: {str(e)[:50]}")
    
    return test_cases


# Export traces from agent log groups
print("="*60)
print("EXPORTING PRODUCTION TRACES")
print("="*60)

if AGENT_LOG_GROUPS:
    PRODUCTION_TRACES = export_traces_from_log_groups(
        log_groups=AGENT_LOG_GROUPS,
        lookback_hours=LOOKBACK_HOURS,
        max_traces=50,
        fetch_tool_calls=True  # Query all events to get actual tool calls
    )
else:
    PRODUCTION_TRACES = []
    print("\n⚠️ No agent log groups found")

print(f"\n{'='*60}")
print(f"📊 Export Summary")
print(f"{'='*60}")
print(f"   Total test cases: {len(PRODUCTION_TRACES)}")

# Count traces with tool calls
traces_with_tools = sum(1 for t in PRODUCTION_TRACES if t.get('tools_called'))
print(f"   Traces with tool calls: {traces_with_tools}")

# Show routing distribution
if PRODUCTION_TRACES:
    routing_counts = {}
    for trace in PRODUCTION_TRACES:
        domain = trace.get('routing_info', {}).get('primary_domain', 'UNKNOWN')
        routing_counts[domain] = routing_counts.get(domain, 0) + 1
    
    print(f"\n   Routing distribution:")
    for domain, count in sorted(routing_counts.items()):
        pct = count / len(PRODUCTION_TRACES) * 100
        print(f"      {domain}: {count} ({pct:.0f}%)")

if PRODUCTION_TRACES:
    print(f"\n📝 Sample test case:")
    sample = PRODUCTION_TRACES[0]
    print(f"   Trace ID: {sample['trace_id'][:20]}...")
    print(f"   Input: {sample['input'][:60]}..." if len(sample['input']) > 60 else f"   Input: {sample['input']}")
    print(f"   Output: {sample['actual_output'][:60]}..." if len(sample['actual_output']) > 60 else f"   Output: {sample['actual_output']}")
    tools = sample.get('tools_called', [])
    if tools:
        print(f"   Tools ({len(tools)}): {[t['name'] for t in tools[:3]]}")
    print(f"   Routing: {sample.get('routing_info', {})}")

EXPORTING PRODUCTION TRACES

Exporting traces from agent runtime log groups...
  Log groups: 34
  Lookback: 24 hours
  Max traces: 50
  Fetch tool calls: True

  ecommerce_workshop_orchestrator_observability-PtyGWx6yGQ-DEFAULT: 36 final response events
    ✓ Trace 697864703198: 1 tools [OrderTools___get_order_status]
    ✓ Trace 69786f5c6191: 1 tools [OrderTools___get_order_status]
    ✓ Trace 69786f690442: 1 tools [OrderTools___track_shipment]
    ✓ Trace 6978722e306e: 1 tools [OrderTools___get_order_status]
    ✓ Trace 69787239732c: 1 tools [OrderTools___track_shipment]
    ✓ Trace 697872411702: 1 tools [ProductTools___search_products]
    ✓ Trace 6978724e5777: 1 tools [ProductTools___get_product_details]
    ✓ Trace 69787257594b: 1 tools [AccountTools___get_membership_benefits]
    ✓ Trace 697872696c44: 3 tools [OrderTools___get_order_status, OrderTools___get_order_status, ProductTools___search_products]
    ✓ Trace 697874e50c8e: 1 tools [OrderTools___get_order_status]
    ✓ Trace 6

## Step 6: Alternative - Export from S3 (Firehose Data)

If Firehose has been running, we can also query from S3 for historical data.

In [ ]:
import gzip

def export_traces_from_s3(
    bucket: str,
    prefix: str = 'traces/',
    lookback_hours: int = 24,
    max_traces: int = 100
) -> List[Dict[str, Any]]:
    """
    Export traces from S3 (Firehose destination).
    
    Args:
        bucket: S3 bucket name
        prefix: S3 prefix for trace files
        lookback_hours: How far back to look
        max_traces: Maximum traces to return
        
    Returns:
        List of test case dicts
    """
    print(f"\nExporting traces from S3...")
    print(f"  Bucket: {bucket}")
    print(f"  Prefix: {prefix}")
    
    test_cases = []
    trace_ids_seen = set()
    
    # Generate prefixes for the lookback period
    now = datetime.now(timezone.utc)
    prefixes_to_check = []
    
    for hours_ago in range(lookback_hours + 1):
        dt = now - timedelta(hours=hours_ago)
        date_prefix = f"{prefix}year={dt.year}/month={dt.month:02d}/day={dt.day:02d}/hour={dt.hour:02d}/"
        prefixes_to_check.append(date_prefix)
    
    try:
        for s3_prefix in prefixes_to_check:
            if len(test_cases) >= max_traces:
                break
            
            try:
                response = s3_client.list_objects_v2(
                    Bucket=bucket,
                    Prefix=s3_prefix,
                    MaxKeys=20
                )
                
                for obj in response.get('Contents', []):
                    if len(test_cases) >= max_traces:
                        break
                    
                    key = obj['Key']
                    
                    # Download and parse file
                    try:
                        obj_response = s3_client.get_object(Bucket=bucket, Key=key)
                        body = obj_response['Body'].read()
                        
                        # Handle gzip compression
                        if key.endswith('.gz'):
                            body = gzip.decompress(body)
                        
                        content = body.decode('utf-8')
                        
                        # Parse each line as JSON
                        for line in content.split('\n'):
                            if not line.strip():
                                continue
                            
                            if len(test_cases) >= max_traces:
                                break
                            
                            otel_event = parse_otel_event(line)
                            if not otel_event:
                                continue
                            
                            if not is_final_response(otel_event):
                                continue
                            
                            trace_id = otel_event.get('traceId', '')
                            if trace_id in trace_ids_seen:
                                continue
                            trace_ids_seen.add(trace_id)
                            
                            test_case = transform_otel_to_test_case(otel_event)
                            if test_case:
                                test_cases.append(test_case)
                                
                    except Exception as e:
                        print(f"  Error reading {key}: {str(e)[:50]}")
                        
            except Exception:
                pass  # Prefix may not exist yet
        
        if test_cases:
            print(f"  ✅ Extracted {len(test_cases)} test cases from S3")
        else:
            print(f"  No traces found in S3 yet (Firehose may need time to buffer)")
            
    except ClientError as e:
        print(f"  S3 bucket not accessible: {str(e)[:50]}")
    
    return test_cases


# Try to get additional traces from S3 if we need more
if len(PRODUCTION_TRACES) < 10:
    print("\n--- Checking S3 for additional traces ---")
    s3_traces = export_traces_from_s3(
        bucket=TRACES_BUCKET,
        prefix='traces/',
        lookback_hours=LOOKBACK_HOURS,
        max_traces=50 - len(PRODUCTION_TRACES)
    )
    
    # Merge unique traces
    existing_ids = {t['trace_id'] for t in PRODUCTION_TRACES}
    for trace in s3_traces:
        if trace['trace_id'] not in existing_ids:
            PRODUCTION_TRACES.append(trace)
    
    print(f"\n📊 Updated total: {len(PRODUCTION_TRACES)} test cases")

## Step 7: Import Custom Evaluators

Import the same evaluators from Module 2 for consistency. These are defined in `custom_evaluators.py`.

In [8]:
import sys
sys.path.insert(0, '.')

from strands_evals import Experiment, Case

# Import custom evaluators from Module 2 style
from custom_evaluators import (
    GoalSuccessEvaluator,
    HelpfulnessEvaluator,
    RoutingAccuracyEvaluator,
    PolicyComplianceEvaluator,
    ResponseQualityEvaluator,
    CustomerSatisfactionEvaluator,
    get_all_custom_evaluators,
)

# Get all evaluators as a dictionary
EVALUATORS = get_all_custom_evaluators()

print(f"✅ Loaded {len(EVALUATORS)} evaluators from custom_evaluators.py:")
for name in EVALUATORS:
    print(f"   • {name}")

✅ Loaded 6 evaluators from custom_evaluators.py:
   • goal_success
   • helpfulness
   • routing_accuracy
   • policy_compliance
   • response_quality
   • customer_satisfaction


## Step 8: Convert to Strands Eval Cases and Run Evaluation

In [9]:
def convert_traces_to_cases(traces: List[Dict[str, Any]]) -> List[Case]:
    """
    Convert production traces to strands-evals Case objects.
    
    Key insight: for production evaluation, we already HAVE the output.
    We don't need to re-invoke the agent - we evaluate the cached output.
    """
    cases = []
    
    for trace in traces:
        case = Case(
            name=trace['trace_id'][:20],
            input=trace['input'],
            expected_output=trace['actual_output'],
            metadata={
                'trace_id': trace['trace_id'],
                'session_id': trace.get('session_id', ''),
                'timestamp': trace['timestamp'],
                'tools_called': trace.get('tools_called', []),
                'routing_info': trace.get('routing_info', {}),
            }
        )
        cases.append(case)
    
    return cases


# Convert traces to cases
if PRODUCTION_TRACES:
    EVAL_CASES = convert_traces_to_cases(PRODUCTION_TRACES)
    print(f"✅ Converted {len(EVAL_CASES)} traces to evaluation cases")
    
    # Create response cache
    RESPONSE_CACHE = {}
    for trace in PRODUCTION_TRACES:
        case_name = trace['trace_id'][:20]
        RESPONSE_CACHE[case_name] = trace['actual_output']
    
    print(f"✅ Cached {len(RESPONSE_CACHE)} production responses")
    
    # Show sample test cases with routing info
    print(f"\n{'='*70}")
    print("SAMPLE TEST CASES WITH ROUTING INFO")
    print(f"{'='*70}")
    
    num_samples = min(3, len(EVAL_CASES))
    for i in range(num_samples):
        case = EVAL_CASES[i]
        trace = PRODUCTION_TRACES[i]
        routing = trace.get('routing_info', {})
        
        print(f"\n{'─'*70}")
        print(f"Test Case {i+1} of {len(EVAL_CASES)}")
        print(f"{'─'*70}")
        print(f"Trace ID:  {case.name}")
        print(f"Timestamp: {case.metadata.get('timestamp', 'N/A')}")
        
        # Show input
        input_display = case.input if len(case.input) <= 80 else case.input[:80] + "..."
        print(f"\n📥 INPUT: {input_display}")
        
        # Show routing information
        print(f"\n🔀 ROUTING:")
        print(f"   Primary Domain: {routing.get('primary_domain', 'UNKNOWN')}")
        
        tools_used = routing.get('tools_used', [])
        if tools_used:
            print(f"   Tools Used ({len(tools_used)}):")
            for tool in tools_used[:5]:  # Show up to 5 tools
                print(f"      • {tool}")
            if len(tools_used) > 5:
                print(f"      ... and {len(tools_used) - 5} more")
        
        sub_agents = routing.get('sub_agents_called', [])
        if sub_agents:
            print(f"   Sub-Agents Called: {', '.join(sub_agents)}")
        
        # Show output (truncate)
        output_display = case.expected_output if len(case.expected_output) <= 120 else case.expected_output[:120] + "..."
        print(f"\n📤 OUTPUT: {output_display}")
    
    print(f"\n{'='*70}")
    print(f"Showing {num_samples} of {len(EVAL_CASES)} test cases")
    print(f"{'='*70}")
    
else:
    EVAL_CASES = []
    RESPONSE_CACHE = {}
    print("\n⚠️ No production traces to convert")
    print("   Please ensure:")
    print("   1. Agents from Module 4a are deployed")
    print("   2. Agents have been invoked (generate some test traffic)")
    print("   3. Wait a few minutes for logs to be available")


def cached_task(case) -> str:
    """Task function that returns cached production response."""
    return RESPONSE_CACHE.get(case.name, "Error: Response not found in cache")

✅ Converted 35 traces to evaluation cases
✅ Cached 35 production responses

SAMPLE TEST CASES WITH ROUTING INFO

──────────────────────────────────────────────────────────────────────
Test Case 1 of 35
──────────────────────────────────────────────────────────────────────
Trace ID:  697864703198b7a1077d
Timestamp: 2026-01-27T07:08:40.442617Z

📥 INPUT: What is the status of order ORD-2024-10001?

🔀 ROUTING:
   Primary Domain: ORDER
   Tools Used (1):
      • OrderTools___get_order_status

📤 OUTPUT: Great news! Here's the status of your order ORD-2024-10001:

**Order Status: DELIVERED** ✅

**Order Details:**
- **Order...

──────────────────────────────────────────────────────────────────────
Test Case 2 of 35
──────────────────────────────────────────────────────────────────────
Trace ID:  69786f5c61913a48341c
Timestamp: 2026-01-27T07:55:19.095559Z

📥 INPUT: What is the status of order ORD-2024-10001?

🔀 ROUTING:
   Primary Domain: ORDER
   Tools Used (1):
      • OrderTools___get_order_

In [10]:
# Run all evaluators on production traces
print("="*60)
print("RUNNING BATCH EVALUATION")
print("="*60)

EVALUATION_RESULTS = {}

if EVAL_CASES:
    for evaluator_name, evaluator in EVALUATORS.items():
        print(f"\n--- Running {evaluator_name} evaluation ---")
        
        experiment = Experiment(
            cases=EVAL_CASES,
            evaluators=[evaluator]
        )
        
        results = experiment.run_evaluations(cached_task)
        report = results[0]
        
        EVALUATION_RESULTS[evaluator_name] = {
            'scores': report.scores,
            'reasons': report.reasons,
            'overall_score': report.overall_score,
            'pass_rate': sum(report.test_passes) / len(report.test_passes) if report.test_passes else 0,
        }
        
        print(f"   Overall Score: {report.overall_score:.2f}")
        print(f"   Pass Rate: {sum(report.test_passes)}/{len(report.test_passes)}")
        
        time.sleep(1)  # Rate limiting
else:
    print("\n⚠️ No evaluation cases to process")
    print("   Generate some agent traffic first, then re-run this notebook")

RUNNING BATCH EVALUATION

--- Running goal_success evaluation ---
   Overall Score: 0.92
   Pass Rate: 31/35

--- Running helpfulness evaluation ---
   Overall Score: 0.93
   Pass Rate: 32/35

--- Running routing_accuracy evaluation ---
   Overall Score: 0.94
   Pass Rate: 32/35

--- Running policy_compliance evaluation ---
   Overall Score: 0.97
   Pass Rate: 34/35

--- Running response_quality evaluation ---
   Overall Score: 0.97
   Pass Rate: 32/35

--- Running customer_satisfaction evaluation ---
   Overall Score: 0.84
   Pass Rate: 32/35


## Step 9: Aggregate and Display Results

In [11]:
print("\n" + "="*60)
print("BATCH EVALUATION SUMMARY")
print("="*60)

if EVALUATION_RESULTS:
    print(f"\nTotal traces evaluated: {len(EVAL_CASES)}")
    print(f"Time range: Last {LOOKBACK_HOURS} hours")
    print(f"\n{'Metric':<25} {'Overall Score':<15} {'Pass Rate':<15}")
    print("-"*55)
    
    for metric_name, result in EVALUATION_RESULTS.items():
        overall = result['overall_score']
        pass_rate = result['pass_rate']
        print(f"{metric_name:<25} {overall:>10.1%}      {pass_rate:>10.1%}")
    
    # Calculate aggregate metrics
    aggregate_metrics = {
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'traces_evaluated': len(EVAL_CASES),
        'lookback_hours': LOOKBACK_HOURS,
    }
    
    for metric_name, result in EVALUATION_RESULTS.items():
        aggregate_metrics[f'{metric_name}_score'] = result['overall_score']
        aggregate_metrics[f'{metric_name}_pass_rate'] = result['pass_rate']
    
    print("\n" + "="*60)
else:
    print("\n⚠️ No evaluation results to display")
    aggregate_metrics = {}


BATCH EVALUATION SUMMARY

Total traces evaluated: 35
Time range: Last 24 hours

Metric                    Overall Score   Pass Rate      
-------------------------------------------------------
goal_success                   92.1%           88.6%
helpfulness                    92.9%           91.4%
routing_accuracy               94.0%           91.4%
policy_compliance              97.1%           97.1%
response_quality               96.6%           91.4%
customer_satisfaction          83.6%           91.4%



In [ ]:
# Create detailed results DataFrame with routing info
if EVALUATION_RESULTS and EVAL_CASES:
    rows = []
    
    for i, case in enumerate(EVAL_CASES):
        trace = PRODUCTION_TRACES[i]
        routing = trace.get('routing_info', {})
        
        row = {
            'trace_id': case.name,
            'timestamp': case.metadata.get('timestamp', ''),
            'input': trace['input'][:80] + '...' if len(trace['input']) > 80 else trace['input'],
            'output': trace['actual_output'][:100] + '...' if len(trace['actual_output']) > 100 else trace['actual_output'],
            'domain': routing.get('primary_domain', 'UNKNOWN'),
            'tools_used': ', '.join(routing.get('tools_used', [])[:3]) or 'None',
            'num_tools': len(routing.get('tools_used', [])),
        }
        
        for metric_name, result in EVALUATION_RESULTS.items():
            row[metric_name] = result['scores'][i] if i < len(result['scores']) else None
        
        rows.append(row)
    
    results_df = pd.DataFrame(rows)
    
    print("📊 Detailed Results with Routing Info (first 10 rows):")
    print("-" * 120)
    
    # Display with better formatting
    pd.set_option('display.max_colwidth', 40)
    display_cols = ['trace_id', 'input', 'output', 'domain', 'tools_used'] + list(EVALUATORS.keys())
    print(results_df[display_cols].head(10).to_string(index=False))
    
    # Routing summary
    print(f"\n📈 Routing Distribution:")
    domain_counts = results_df['domain'].value_counts()
    for domain, count in domain_counts.items():
        pct = count / len(results_df) * 100
        print(f"   {domain}: {count} ({pct:.1f}%)")
    
    print(f"\n📈 Score Distribution:")
    for metric_name in EVALUATORS.keys():
        if metric_name in results_df.columns:
            scores = results_df[metric_name].dropna()
            if len(scores) > 0:
                print(f"   {metric_name}: mean={scores.mean():.2f}, min={scores.min():.2f}, max={scores.max():.2f}")
else:
    results_df = pd.DataFrame()

📊 Detailed Results with Routing Info (first 5 rows):
------------------------------------------------------------------------------------------------------------------------
            trace_id                                                            input                                                                                                      output  domain                                                    tools_used  goal_success  helpfulness  routing_accuracy  policy_compliance  response_quality  customer_satisfaction
697864703198b7a1077d                      What is the status of order ORD-2024-10001? Great news! Here's the status of your order ORD-2024-10001:\n\n**Order Status: DELIVERED** ✅\n\n**Order ...   ORDER                                 OrderTools___get_order_status          1.00         1.00               1.0                1.0               1.0                   1.00
69786f5c61913a48341c                      What is the status of order ORD-2024-10001? Gr

## Step 10: Store Results

In [13]:
# Save results locally
RESULTS_DIR = 'evaluation_results'
os.makedirs(RESULTS_DIR, exist_ok=True)

timestamp_str = datetime.now().strftime('%Y%m%d_%H%M%S')

if EVALUATION_RESULTS:
    # Build comprehensive results with all data together
    full_results = []
    for i, case in enumerate(EVAL_CASES):
        trace = PRODUCTION_TRACES[i]
        routing = trace.get('routing_info', {})
        
        result_entry = {
            # Trace metadata
            'trace_id': trace['trace_id'],
            'session_id': trace.get('session_id', ''),
            'timestamp': trace['timestamp'],
            
            # Input/Output data used in evaluation
            'input': trace['input'],
            'actual_output': trace['actual_output'],
            
            # Routing and tool use information
            'routing': {
                'primary_domain': routing.get('primary_domain', 'UNKNOWN'),
                'tools_used': routing.get('tools_used', []),
                'sub_agents_called': routing.get('sub_agents_called', []),
                'num_tools': len(routing.get('tools_used', [])),
            },
            'tools_called': trace.get('tools_called', []),
            
            # Evaluation results for each metric
            'evaluations': {}
        }
        
        for metric_name, result in EVALUATION_RESULTS.items():
            if i < len(result['scores']):
                result_entry['evaluations'][metric_name] = {
                    'score': result['scores'][i],
                    'reason': result['reasons'][i] if i < len(result['reasons']) else ''
                }
        
        full_results.append(result_entry)
    
    # Save comprehensive JSON file
    full_file = f"{RESULTS_DIR}/batch_eval_full_{timestamp_str}.json"
    with open(full_file, 'w') as f:
        json.dump(full_results, f, indent=2, default=str)
    print(f"✅ Saved comprehensive results to {full_file}")
    
    # Save aggregate metrics
    aggregate_file = f"{RESULTS_DIR}/batch_eval_aggregate_{timestamp_str}.json"
    with open(aggregate_file, 'w') as f:
        json.dump(aggregate_metrics, f, indent=2, default=str)
    print(f"✅ Saved aggregate metrics to {aggregate_file}")
    
    # Save detailed CSV for easy viewing
    if not results_df.empty:
        detailed_file = f"{RESULTS_DIR}/batch_eval_detailed_{timestamp_str}.csv"
        results_df.to_csv(detailed_file, index=False)
        print(f"✅ Saved detailed CSV to {detailed_file}")
    
    # Display sample of comprehensive results
    print(f"\n{'='*70}")
    print("COMPREHENSIVE EVALUATION RESULTS (Sample)")
    print(f"{'='*70}")
    
    num_samples = min(2, len(full_results))
    for i in range(num_samples):
        entry = full_results[i]
        routing = entry.get('routing', {})
        
        print(f"\n{'─'*70}")
        print(f"TRACE {i+1}: {entry['trace_id'][:30]}...")
        print(f"{'─'*70}")
        
        # Input
        input_display = entry['input'] if len(entry['input']) <= 70 else entry['input'][:70] + "..."
        print(f"\n📥 INPUT: {input_display}")
        
        # Routing info
        print(f"\n🔀 ROUTING:")
        print(f"   Domain: {routing.get('primary_domain', 'UNKNOWN')}")
        tools = routing.get('tools_used', [])
        if tools:
            tools_display = ', '.join(tools[:3])
            if len(tools) > 3:
                tools_display += f" (+{len(tools)-3} more)"
            print(f"   Tools: {tools_display}")
        
        # Output
        output_display = entry['actual_output'] if len(entry['actual_output']) <= 100 else entry['actual_output'][:100] + "..."
        print(f"\n📤 OUTPUT: {output_display}")
        
        # Evaluation scores and reasons
        print(f"\n📊 EVALUATION RESULTS:")
        for metric_name, eval_data in entry['evaluations'].items():
            score = eval_data['score']
            reason = eval_data['reason']
            
            # Score indicator
            if score >= 0.8:
                indicator = "✅"
            elif score >= 0.5:
                indicator = "⚠️"
            else:
                indicator = "❌"
            
            print(f"\n   {indicator} {metric_name}: {score:.0%}")
            
            # Show reason (truncate if too long)
            if reason:
                reason_display = reason if len(reason) <= 80 else reason[:80] + "..."
                print(f"      Reason: {reason_display}")
    
    print(f"\n{'='*70}")
    print(f"Full results saved to: {full_file}")
    print(f"{'='*70}")
    
else:
    full_results = []
    print("⚠️ No results to save")

✅ Saved comprehensive results to evaluation_results/batch_eval_full_20260127_231050.json
✅ Saved aggregate metrics to evaluation_results/batch_eval_aggregate_20260127_231050.json
✅ Saved detailed CSV to evaluation_results/batch_eval_detailed_20260127_231050.csv

COMPREHENSIVE EVALUATION RESULTS (Sample)

──────────────────────────────────────────────────────────────────────
TRACE 1: 697864703198b7a1077d9db96fb17b...
──────────────────────────────────────────────────────────────────────

📥 INPUT: What is the status of order ORD-2024-10001?

🔀 ROUTING:
   Domain: ORDER
   Tools: OrderTools___get_order_status

📤 OUTPUT: Great news! Here's the status of your order ORD-2024-10001:

**Order Status: DELIVERED** ✅

**Order ...

📊 EVALUATION RESULTS:

   ✅ goal_success: 100%
      Reason: The output perfectly matches the expected output in all aspects. The agent fully...

   ✅ helpfulness: 100%
      Reason: The response is extremely helpful - it provides comprehensive order information ...

  

In [ ]:
# Publish metrics to CloudWatch
def publish_metrics_to_cloudwatch(metrics: Dict[str, Any]) -> bool:
    """Publish evaluation metrics to CloudWatch."""
    if not metrics:
        return False
    
    metric_data = []
    timestamp = datetime.now(timezone.utc)
    
    metric_data.append({
        'MetricName': 'TracesEvaluated',
        'Value': metrics.get('traces_evaluated', 0),
        'Unit': 'Count',
        'Timestamp': timestamp,
    })
    
    for key, value in metrics.items():
        if key.endswith('_score') and isinstance(value, (int, float)):
            metric_name = key.replace('_score', '_Score')
            metric_data.append({
                'MetricName': metric_name,
                'Value': value * 100,
                'Unit': 'Percent',
                'Timestamp': timestamp,
            })
        elif key.endswith('_pass_rate') and isinstance(value, (int, float)):
            metric_name = key.replace('_pass_rate', '_PassRate')
            metric_data.append({
                'MetricName': metric_name,
                'Value': value * 100,
                'Unit': 'Percent',
                'Timestamp': timestamp,
            })
    
    if metric_data:
        try:
            for i in range(0, len(metric_data), 20):
                batch = metric_data[i:i+20]
                cloudwatch_client.put_metric_data(
                    Namespace=CLOUDWATCH_NAMESPACE,
                    MetricData=batch
                )
            print(f"✅ Published {len(metric_data)} metrics to CloudWatch namespace: {CLOUDWATCH_NAMESPACE}")
            return True
        except Exception as e:
            print(f"❌ Error publishing to CloudWatch: {e}")
            return False
    
    return False


if aggregate_metrics:
    publish_metrics_to_cloudwatch(aggregate_metrics)

In [ ]:
# Save to S3
def save_results_to_s3(results: List[Dict], bucket: str, prefix: str = 'batch-evaluations') -> str:
    """Save evaluation results to S3."""
    timestamp_str = datetime.now().strftime('%Y/%m/%d/%H')
    key = f"{prefix}/{timestamp_str}/batch_eval_{datetime.now().strftime('%Y%m%d%H%M%S')}.json"
    
    try:
        s3_client.put_object(
            Bucket=bucket,
            Key=key,
            Body=json.dumps(results, indent=2, default=str).encode('utf-8'),
            ContentType='application/json'
        )
        
        print(f"✅ Saved results to s3://{bucket}/{key}")
        return f"s3://{bucket}/{key}"
        
    except Exception as e:
        print(f"❌ Error saving to S3: {e}")
        return ""


if EVALUATION_RESULTS and full_results:
    s3_path = save_results_to_s3(full_results, RESULTS_BUCKET)
else:
    s3_path = ""

## Step 11: Summary

In [ ]:
print("\n" + "="*70)
print("MODULE 5 COMPLETE: PRODUCTION BATCH EVALUATION")
print("="*70)

print("\n📋 WHAT WE ACCOMPLISHED")
print("-"*70)

print("\n1. STREAMING INFRASTRUCTURE")
print(f"   ✅ S3 bucket for traces: {TRACES_BUCKET}")
print(f"   ✅ Kinesis Firehose stream: {FIREHOSE_STREAM_NAME}")
print(f"   ✅ Subscription filters on {len(AGENT_LOG_GROUPS)} agent log groups")

print("\n2. TRACE EXPORT")
print(f"   ✅ Exported {len(PRODUCTION_TRACES)} traces from agent runtime logs")
print(f"   ✅ Time range: Last {LOOKBACK_HOURS} hours")

print("\n3. BATCH EVALUATION")
if EVALUATION_RESULTS:
    print(f"   ✅ Ran {len(EVALUATORS)} evaluators on {len(EVAL_CASES)} traces")
    print("   ✅ Evaluators: Goal Success, Helpfulness, Routing, Policy, Quality, Satisfaction")
else:
    print("   ⚠️ No evaluations run (no traces available)")

print("\n4. RESULTS STORAGE")
print(f"   ✅ Local files: {RESULTS_DIR}/")
print(f"   ✅ CloudWatch metrics: {CLOUDWATCH_NAMESPACE}")
if s3_path:
    print(f"   ✅ S3: {s3_path}")

print("\n" + "="*70)
print("KEY TAKEAWAYS")
print("="*70)
print("""
1. OTEL Events Location:
   - NOT in aws/spans - that's for X-Ray style traces
   - In agent runtime log groups: /aws/bedrock-agentcore/runtimes/*
   - Filter: scope.name = "strands.telemetry.tracer"

2. Streaming Pipeline Architecture:
   - CloudWatch Logs → Subscription Filter → Firehose → S3
   - Enables real-time trace capture and batch processing
   - Similar to production-grade evaluation pipelines

3. Response Caching:
   - Production traces ALREADY contain responses
   - No need to re-invoke agents during evaluation
   - Enables efficient batch processing

4. Next Steps:
   - Schedule this notebook to run periodically
   - Set up CloudWatch alarms on evaluation metrics
   - Use Athena to query historical traces in S3
""")
print("="*70)

In [ ]:
# Store variables for use in other modules
%store aggregate_metrics
%store EVALUATION_RESULTS
%store REGION
%store TRACES_BUCKET
%store FIREHOSE_STREAM_NAME

print("\n✅ Variables stored for use in subsequent modules")

In [ ]:
# Console links
print("\n" + "="*70)
print("AWS CONSOLE LINKS")
print("="*70)

print(f"\n📊 Batch Evaluation Metrics:")
print(f"   https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#metricsV2:graph=~();namespace={CLOUDWATCH_NAMESPACE.replace('/', '%2F')}")

print(f"\n🔥 Kinesis Firehose Stream:")
print(f"   https://{REGION}.console.aws.amazon.com/firehose/home?region={REGION}#/details/{FIREHOSE_STREAM_NAME}")

print(f"\n📦 S3 Traces Bucket:")
print(f"   https://s3.console.aws.amazon.com/s3/buckets/{TRACES_BUCKET}?region={REGION}")

print(f"\n📝 Agent Runtime Log Groups:")
for lg in AGENT_LOG_GROUPS[:3]:
    encoded_lg = lg.replace('/', '%2F')
    print(f"   https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#logsV2:log-groups/log-group/{encoded_lg}")